In [1]:

%load_ext autoreload
%autoreload 2

In [2]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

from wrf_io import *
from scipy.io import savemat
from MITRotor import IEA10MW
from itertools import product
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler

## Train data

In [3]:
save_path = '/scratch/09909/smata/induction_modeling/gaussian_process/10MW/train_data/'

In [4]:
casenames = [
r'sn030_vn200',
r'sn030_vn100',
r'sn030_vn050',
r'sn030_vn025',
r'sn030_v000',
r'sn030_v025',
r'sn030_v050',
r'sn030_v100',
r'sn030_v200',
r'sn020_vn200',
r'sn020_vn100',
r'sn020_vn050',
r'sn020_vn025',
r'sn020_v000',
r'sn020_v025',
r'sn020_v050',
r'sn020_v100',
r'sn020_v200',
r'sn010_vn200',
r'sn010_vn100',
r'sn010_vn050',
r'sn010_vn025',
r'sn010_v000',
r'sn010_v025',
r'sn010_v050',
r'sn010_v100',
r'sn010_v200',
r's000_vn200',
r's000_vn100',
r's000_vn050',
r's000_vn025',
r's000_v000',
r's000_v025',
r's000_v050',
r's000_v100',
r's000_v200',
r's010_vn200',
r's010_vn100',
r's010_vn050',
r's010_vn025',
r's010_v000',
r's010_v025',
r's010_v050',
r's010_v100',
r's010_v200',
r's020_vn200',
r's020_vn100',
r's020_vn050',
r's020_vn025',
r's020_v000',
r's020_v025',
r's020_v050',
r's020_v100',
r's020_v200',
r's030_vn200',
r's030_vn100',
r's030_vn050',
r's030_vn025',
r's030_v000',
r's030_v025',
r's030_v050',
r's030_v100',
r's030_v200'
]

In [5]:
par_path = '/scratch/09909/smata/wrf_les_sweep/runs/10MW/rate/gad_sweep/parameters.pkl'

params = postproc.load_params(par_path)
data   = postproc.extract_sounding(params=params, local=False)
wrfles = postproc.load_data(params=params, casenames=casenames, local=False)

allocation     : ATM170028
partition      : spr
runtime        : 48:00:00
system         : stampede
num_nodes      : 2
exclude_time   : 10
save_interval  : 10
base_dir       : /scratch/09909/smata/wrf_les_sweep/runs/10MW/rate
wrf_path       : /work2/09909/smata/stampede3/WRF_LES
template_path  : /scratch/09909/smata/wrf_les_sweep/templates
turb_model     : iea10MW
rotor_model    : GAD
slice_loc      : 1
print_table    : True
plot_outer     : True
save_outer     : False
outer_align    : False
plot_inner     : True
save_inner     : False
save_both      : False
outer_pad      : 75
plot_profiles  : True
save_profiles  : True
batch_submit   : True
prof_type      : Idealized
shear_type     : Rate
shear          : [-0.03, -0.02, -0.01, 0, 0.01, 0.02, 0.03]
veer           : [-0.2, -0.1, -0.05, -0.025, 0, 0.025, 0.05, 0.1, 0.2]
excluded_pairs : []
Ufst           : 7


In [65]:
fontsize = 24
plt.rcParams['xtick.labelsize'] = 20 
plt.rcParams['ytick.labelsize'] = 20 

plt.rcParams.update({
    'text.usetex': True,
    'text.latex.preamble': r'\usepackage{amsfonts}'
})

In [6]:
cases = [pair for pair in product(params['shear'], params['veer'])]
shears = [pair[0] for pair in cases]
veers = [pair[1] for pair in cases]

shears = np.array(shears)  
veers  = np.array(veers)

In [7]:
Nelm = wrfles[0]['Nelm']
Nsct = wrfles[0]['Nsct']

t = np.linspace(0,2*np.pi,Nsct)
r = np.linspace(0,1,Nelm)

rho = 1.225

z_hh = wrfles[0]['hub_height']

R, T = np.meshgrid(r, t)

X = R * np.sin(T)
Y = (R * np.cos(T)) * wrfles[0]['radius'] + z_hh

wrf_vax      = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_vtn      = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_vax_real = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_vtn_real = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_vtn_NR   = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_vtn_NR_real= np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_W        = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_W_real   = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_CL       = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_CD       = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_CL_real  = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_CD_real  = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_L        = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_D        = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_L_real   = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_D_real   = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_FN       = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_FT       = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_FN_real  = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_FT_real  = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_phi      = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_phi_real = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_aoa      = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_aoa_real = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_cax      = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_pow      = np.zeros(len(casenames),dtype='longdouble')
wrf_thr      = np.zeros(len(casenames),dtype='longdouble')

wrf_pow_real = np.zeros(len(casenames),dtype='longdouble')
wrf_thr_real = np.zeros(len(casenames),dtype='longdouble')

shapiro      = np.zeros(len(casenames), dtype=float)

Uhub         = np.zeros(len(casenames), dtype=float)

wrf_tsr      = np.zeros(len(casenames), dtype=float)
wrf_omg      = np.zeros(len(casenames), dtype=float)

wrf_cot_rot  = np.zeros(len(casenames), dtype=float)
wrf_ind_rot  = np.zeros(len(casenames), dtype=float)

r_ann        = np.zeros((Nelm, len(casenames)),dtype='longdouble')
shears_ann   = np.zeros((Nelm, len(casenames)),dtype='longdouble')
veers_ann    = np.zeros((Nelm, len(casenames)),dtype='longdouble')

wrf_cot_ann  = np.zeros((Nelm, len(casenames)),dtype='longdouble')
wrf_ind_ann  = np.zeros((Nelm, len(casenames)),dtype='longdouble')

wrf_cot_loc  = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_ind_loc  = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

rotor = IEA10MW()

for count,case in enumerate(casenames):

    # Rotor radius
    R = wrfles[count]['radius']

    # Extract background flow
    u_func = interp1d(data[case][:,0], data[case][:,1], kind='linear')
    v_func = interp1d(data[case][:,0], data[case][:,2], kind='linear')

    # u and v velocity functions
    u_inf = u_func(Y)
    v_inf = v_func(Y)

    # Extract u and v velocity components at the rotor disk
    u_rotor = np.mean(wrfles[count]['shapiroM'][0] * wrfles[count]['u'],axis=0)
    # u_rotor = np.mean(wrfles[count]['u'],axis=0)
    v_rotor = np.mean(wrfles[count]['v'],axis=0)
    w_rotor = np.mean(wrfles[count]['w'],axis=0)

    # Magnitude of wind speed at rotor disk
    U = np.sqrt(u_rotor**2 + v_rotor**2)

    # Local wind misalignment angle at the rotor disk
    wdir = np.atan2(v_rotor,u_rotor)

    # Nondimensional radial positions
    r = wrfles[count]['rOverR']
    r_mat =  (np.ones_like(wdir.T) * r).T

    # Azimuthal coordinates
    theta = np.linspace(0, 2*np.pi, wrfles[count]['Nsct'])

    # Freestream wind speed at hub height
    U_hub = np.sqrt(u_func(z_hh)**2 + v_func(z_hh)**2)

    # Radial chord lengths
    chord = rotor.chord_func(r)
    chord =  (np.ones_like(wdir.T) * chord).T

    # Radial twist
    twist = rotor.twist_func(r)
    
    # Local solidity
    sigma_r = 3 * chord / (2 * np.pi * r_mat * R)

    # Local solidity
    sigma = 3/Nsct

    # Tip speed ratio
    omega = np.mean(wrfles[count]['omega'], axis=0) * 2 * np.pi / 60
    tsr = np.mean(omega*R/(U_hub), axis=0) 

    trbYaw = np.mean(wrfles[count]['yaw'], axis=0) 

    Vax, Vtn_NR, _ = postproc.rotGlobalToLocal(Nelm,Nsct,u_rotor,v_rotor,np.zeros_like(u_rotor))

    Vtn = omega * r_mat * R - Vtn_NR

    # Relative velocity
    W = np.sqrt(Vax**2 + Vtn**2)

    # Inflow angle
    phi = np.atan2(Vax,Vtn)

    aoa  = phi - twist[:, np.newaxis]
    Cl, Cd = rotor.clcd(r_mat, aoa)

    # Axial coefficient
    Cax = Cl * np.cos(phi) + Cd * np.sin(phi)

    # Local CT
    ct = sigma_r * (W/U_hub)**2 * Cax

    L = 1/2 * rho * chord * (Cl * W**2)
    D = 1/2 * rho * chord * (Cd * W**2)

    FN = L * np.cos(phi) + D * np.sin(phi)
    FT = L * np.sin(phi) - D * np.cos(phi)

    dr = (R - rotor.hub_radius)/Nelm

    T = np.sum(FN * dr * sigma)
    P = np.sum(FT * r_mat * R * dr * sigma * omega)

    wrf_vax[:,:,count]      = Vax
    wrf_vtn[:,:,count]      = Vtn

    wrf_vax_real[:,:,count] = np.mean(wrfles[count]['v1'], axis=0)
    wrf_vtn_real[:,:,count] = np.mean(wrfles[count]['v_tan'], axis=0)
  
    wrf_vtn_NR[:,:,count]   = Vtn_NR
    wrf_vtn_NR_real[:,:,count] = np.mean(wrfles[count]['v_tan_no_rot'], axis=0)

    wrf_phi[:,:,count]      = phi
    wrf_phi_real[:,:,count] = np.deg2rad(np.mean(wrfles[count]['phi'], axis=0))

    wrf_aoa[:,:,count]      = aoa
    wrf_aoa_real[:,:,count] = np.deg2rad(np.mean(wrfles[count]['aoa'], axis=0))

    wrf_W[:,:,count]        = W
    wrf_W_real[:,:,count]   = np.mean(wrfles[count]['vrel'], axis=0)

    wrf_cax[:,:,count]      = Cax

    wrf_cot_rot[count]      = postproc.rotor_average(ct,r,theta)
    wrf_ind_rot[count]      = postproc.rotor_average(1 - u_rotor/ U_hub,r,theta)

    wrf_cot_ann[:,count]    = postproc.annulus_average(ct,theta)
    wrf_ind_ann[:,count]    = postproc.annulus_average(1 - u_rotor/ U_hub,theta)

    r_ann[:,count]          = r
    shears_ann[:,count]     = shears[count] * np.ones_like(r)
    veers_ann[:,count]      = veers[count] * np.ones_like(r)

    wrf_cot_loc[:,:,count]  = ct
    wrf_ind_loc[:,:,count]  = 1 - u_rotor/ U_hub

    wrf_CL[:,:,count]       = Cl
    wrf_CD[:,:,count]       = Cd

    wrf_CL_real[:,:,count]  = np.mean(wrfles[count]['cl'], axis=0)
    wrf_CD_real[:,:,count]  = np.mean(wrfles[count]['cd'], axis=0)

    wrf_L[:,:,count]        = L
    wrf_D[:,:,count]        = D

    wrf_L_real[:,:,count]   = np.mean(wrfles[count]['l'], axis=0)
    wrf_D_real[:,:,count]   = np.mean(wrfles[count]['d'], axis=0)

    wrf_FN[:,:,count]       = FN
    wrf_FT[:,:,count]       = FT

    wrf_FN_real[:,:,count]  = np.mean(wrfles[count]['fn'], axis=0)
    wrf_FT_real[:,:,count]  = np.mean(wrfles[count]['ft'], axis=0)

    wrf_thr[count]          = T
    wrf_pow[count]          = P

    wrf_thr_real[count]     = np.mean(wrfles[count]['thrust'], axis=0)[0]
    wrf_pow_real[count]     = np.mean(wrfles[count]['power_aero'], axis=0)[0]

    Uhub[count]             = U_hub
    wrf_tsr[count]          = tsr
    wrf_omg[count]          = omega[0]

# END LOOP

np.save(save_path + 'wrf_cot_loc.npy', wrf_cot_loc)
np.save(save_path + 'wrf_ind_loc.npy', wrf_ind_loc)

np.save(save_path + 'wrf_cot_ann.npy', wrf_cot_ann)
np.save(save_path + 'wrf_ind_ann.npy', wrf_ind_ann)

np.save(save_path + 'wrf_cot_rot.npy', wrf_cot_rot)
np.save(save_path + 'wrf_ind_rot.npy', wrf_ind_rot)
np.savez(save_path + "rotor_dims_10MW.npz", Nsct=Nsct, Nelm=Nelm, rOverR=r, R = R)

In [68]:
# Get indices where values are negative
neg_coords = np.argwhere(wrf_ind_ann < 0)

# Print results
print(f"Found {len(neg_coords)} negative value(s) in wrf_ind_ann:")
for i, j in neg_coords:
    print(f"  At ({i}, {j}): {wrf_ind_ann[i, j]}")

Found 2 negative value(s) in wrf_ind_ann:
  At (1, 4): -0.0015690295307364045
  At (1, 58): -0.007218356669694954


In [8]:
# Create a list of (name, array) pairs
variables = [
    ('r_ann'      , r_ann),
    ('wrf_cot_loc', wrf_cot_loc),
    ('wrf_ind_loc', wrf_ind_loc),
    ('wrf_cot_ann', wrf_cot_ann),
    ('wrf_ind_ann', wrf_ind_ann),
    ('wrf_cot_rot', wrf_cot_rot),
    ('wrf_ind_rot', wrf_ind_rot),
    ('shears'     , shears),
    ('veers'      , veers),
    ('shears_ann' , shears_ann),
    ('veers_ann'  , veers_ann),
]

# Print descriptive statistics
print("Variable Statistics:")
print("-" * 75)
print(f"{'Name':<15} | {'Mean':>8} | {'Std Dev':>8} | {'Min':>8} | {'Max':>8}")
print("-" * 75)

for name, arr in variables:
    arr_flat = arr.flatten()
    mean = np.mean(arr_flat)
    std = np.std(arr_flat)
    min_val = np.min(arr_flat)
    max_val = np.max(arr_flat)
    print(f"{name:<15} | {mean:>8.4f} | {std:>8.4f} | {min_val:>8.4f} | {max_val:>8.4f}")

Variable Statistics:
---------------------------------------------------------------------------
Name            |     Mean |  Std Dev |      Min |      Max
---------------------------------------------------------------------------
r_ann           |   0.5121 |   0.2815 |   0.0429 |   0.9812
wrf_cot_loc     |   0.7118 |   0.3095 |   0.0733 |   1.1959
wrf_ind_loc     |   0.2390 |   0.1541 |  -0.3718 |   0.6396
wrf_cot_ann     |   0.7118 |   0.3004 |   0.1031 |   1.0409
wrf_ind_ann     |   0.2390 |   0.1089 |  -0.0072 |   0.3579
wrf_cot_rot     |   0.6873 |   0.0069 |   0.6705 |   0.7022
wrf_ind_rot     |   0.2439 |   0.0100 |   0.2218 |   0.2676
shears          |   0.0000 |   0.0200 |  -0.0300 |   0.0300
veers           |  -0.0000 |   0.1087 |  -0.2000 |   0.2000
shears_ann      |   0.0000 |   0.0200 |  -0.0300 |   0.0300
veers_ann       |   0.0000 |   0.1087 |  -0.2000 |   0.2000


In [9]:
# Generate MATLAB tables of standardized inputs

X_rot = np.column_stack([wrf_cot_rot, shears, veers])
X_ann = np.column_stack([r_ann.flatten(), wrf_cot_ann.flatten(), shears_ann.flatten(), veers_ann.flatten()])

y_rot = wrf_ind_rot
y_ann = wrf_ind_ann.flatten()

savemat(save_path + 'wrf_10MW_ann_RAW.mat', {'X': X_ann, 'y': y_ann.reshape(-1, 1)})
savemat(save_path + 'wrf_10MW_rot_RAW.mat', {'X': X_rot, 'y': y_rot.reshape(-1, 1)})

In [10]:
# ── standardize every array, overwriting in place ──────────────────────────────
# 1-D (_rot) -------------------------------------------------------------------
scaler_wrf_cot_rot = StandardScaler()
wrf_cot_rot = scaler_wrf_cot_rot.fit_transform(wrf_cot_rot[:, None]).ravel()

scaler_wrf_ind_rot = StandardScaler()
wrf_ind_rot = scaler_wrf_ind_rot.fit_transform(wrf_ind_rot[:, None]).ravel()

scaler_shears = StandardScaler()
shears = scaler_shears.fit_transform(shears[:, None]).ravel()

scaler_veers = StandardScaler()
veers = scaler_veers.fit_transform(veers[:, None]).ravel()

# 2-D (_ann) -------------------------------------------------------------------
scaler_wrf_cot_ann = StandardScaler()
wrf_cot_ann = scaler_wrf_cot_ann.fit_transform(wrf_cot_ann.flatten()[:, None]) \
                                .reshape(wrf_cot_ann.shape)

scaler_wrf_ind_ann = StandardScaler()
wrf_ind_ann = scaler_wrf_ind_ann.fit_transform(wrf_ind_ann.flatten()[:, None]) \
                                .reshape(wrf_ind_ann.shape)

scaler_shears_ann = StandardScaler()
shears_ann = scaler_shears_ann.fit_transform(shears_ann.flatten()[:, None]) \
                              .reshape(shears_ann.shape)

scaler_veers_ann = StandardScaler()
veers_ann = scaler_veers_ann.fit_transform(veers_ann.flatten()[:, None]) \
                            .reshape(veers_ann.shape)

In [11]:
# Create a list of (name, array) pairs
variables = [
    ('r_ann'      , r_ann),
    ('wrf_cot_loc', wrf_cot_loc),
    ('wrf_ind_loc', wrf_ind_loc),
    ('wrf_cot_ann', wrf_cot_ann),
    ('wrf_ind_ann', wrf_ind_ann),
    ('wrf_cot_rot', wrf_cot_rot),
    ('wrf_ind_rot', wrf_ind_rot),
    ('shears'     , shears),
    ('veers'      , veers),
    ('shears_ann' , shears_ann),
    ('veers_ann'  , veers_ann),
]

# Print descriptive statistics
print("Variable Statistics:")
print("-" * 75)
print(f"{'Name':<15} | {'Mean':>8} | {'Std Dev':>8} | {'Min':>8} | {'Max':>8}")
print("-" * 75)

for name, arr in variables:
    arr_flat = arr.flatten()
    mean = np.mean(arr_flat)
    std = np.std(arr_flat)
    min_val = np.min(arr_flat)
    max_val = np.max(arr_flat)
    print(f"{name:<15} | {mean:>8.4f} | {std:>8.4f} | {min_val:>8.4f} | {max_val:>8.4f}")

Variable Statistics:
---------------------------------------------------------------------------
Name            |     Mean |  Std Dev |      Min |      Max
---------------------------------------------------------------------------
r_ann           |   0.5121 |   0.2815 |   0.0429 |   0.9812
wrf_cot_loc     |   0.7118 |   0.3095 |   0.0733 |   1.1959
wrf_ind_loc     |   0.2390 |   0.1541 |  -0.3718 |   0.6396
wrf_cot_ann     |   0.0000 |   1.0000 |  -2.0263 |   1.0955
wrf_ind_ann     |  -0.0000 |   1.0000 |  -2.2621 |   1.0916
wrf_cot_rot     |   0.0000 |   1.0000 |  -2.4505 |   2.1752
wrf_ind_rot     |   0.0000 |   1.0000 |  -2.1992 |   2.3643
shears          |   0.0000 |   1.0000 |  -1.5000 |   1.5000
veers           |  -0.0000 |   1.0000 |  -1.8407 |   1.8407
shears_ann      |   0.0000 |   1.0000 |  -1.5000 |   1.5000
veers_ann       |   0.0000 |   1.0000 |  -1.8407 |   1.8407


In [12]:
# Save scalers

with open(os.path.join(save_path, 'scaler_wrf_cot_ann.pkl'), 'wb') as f:
    pickle.dump(scaler_wrf_cot_ann, f)

with open(os.path.join(save_path, 'scaler_wrf_ind_ann.pkl'), 'wb') as f:
    pickle.dump(scaler_wrf_ind_ann, f)

with open(os.path.join(save_path, 'scaler_wrf_cot_rot.pkl'), 'wb') as f:
    pickle.dump(scaler_wrf_cot_rot, f)

with open(os.path.join(save_path, 'scaler_wrf_ind_rot.pkl'), 'wb') as f:
    pickle.dump(scaler_wrf_ind_rot, f)

with open(os.path.join(save_path, 'scaler_shears_rot.pkl'), 'wb') as f:
    pickle.dump(scaler_shears, f)

with open(os.path.join(save_path, 'scaler_veers_rot.pkl'), 'wb') as f:
    pickle.dump(scaler_veers, f)

with open(os.path.join(save_path, 'scaler_shears_ann.pkl'), 'wb') as f:
    pickle.dump(scaler_shears_ann, f)

with open(os.path.join(save_path, 'scaler_veers_ann.pkl'), 'wb') as f:
    pickle.dump(scaler_veers_ann, f)

In [13]:
# Generate MATLAB inputs for GP search

X_rot = np.column_stack([wrf_cot_rot, shears, veers])
X_ann = np.column_stack([r_ann.flatten(), wrf_cot_ann.flatten(), shears_ann.flatten(), veers_ann.flatten()])

y_rot = wrf_ind_rot
y_ann = wrf_ind_ann.flatten()

savemat(save_path + 'wrf_10MW_ann.mat', {'X': X_ann, 'y': y_ann.reshape(-1, 1)})
savemat(save_path + 'wrf_10MW_rot.mat', {'X': X_rot, 'y': y_rot.reshape(-1, 1)})

## Test data

In [14]:
save_path = '/scratch/09909/smata/induction_modeling/gaussian_process/10MW/test_data/'

In [15]:
casenames = [
r'sn035_vn250',
r'sn035_vn150',
r'sn035_vn075',
r'sn035_vn035',
r'sn035_vn010',
r'sn035_v010',
r'sn035_v035',
r'sn035_v075',
r'sn035_v150',
r'sn035_v250',
r'sn025_vn250',
r'sn025_vn150',
r'sn025_vn075',
r'sn025_vn035',
r'sn025_vn010',
r'sn025_v010',
r'sn025_v035',
r'sn025_v075',
r'sn025_v150',
r'sn025_v250',
r'sn015_vn250',
r'sn015_vn150',
r'sn015_vn075',
r'sn015_vn035',
r'sn015_vn010',
r'sn015_v010',
r'sn015_v035',
r'sn015_v075',
r'sn015_v150',
r'sn015_v250',
r'sn005_vn250',
r'sn005_vn150',
r'sn005_vn075',
r'sn005_vn035',
r'sn005_vn010',
r'sn005_v010',
r'sn005_v035',
r'sn005_v075',
r'sn005_v150',
r'sn005_v250',
r's000_vn250',
r's000_vn150',
r's000_vn075',
r's000_vn035',
r's000_vn010',
r's000_v010',
r's000_v035',
r's000_v075',
r's000_v150',
r's000_v250',
r's005_vn250',
r's005_vn150',
r's005_vn075',
r's005_vn035',
r's005_vn010',
r's005_v010',
r's005_v035',
r's005_v075',
r's005_v150',
r's005_v250',
r's015_vn250',
r's015_vn150',
r's015_vn075',
r's015_vn035',
r's015_vn010',
r's015_v010',
r's015_v035',
r's015_v075',
r's015_v150',
r's015_v250',
r's025_vn250',
r's025_vn150',
r's025_vn075',
r's025_vn035',
r's025_vn010',
r's025_v010',
r's025_v035',
r's025_v075',
r's025_v150',
r's025_v250',
r's035_vn250',
r's035_vn150',
r's035_vn075',
r's035_vn035',
r's035_vn010',
r's035_v010',
r's035_v035',
r's035_v075',
r's035_v150',
r's035_v250'
]

In [16]:
par_path = '/scratch/09909/smata/wrf_les_sweep/runs/10MW/rate/OOS_SV/gad_sweep/opt_params.pkl'

params = postproc.load_params(par_path)
data   = postproc.extract_sounding(params=params, local=False)
wrfles = postproc.load_data(params=params, casenames=casenames, local=False)

allocation    : ATM170028
partition     : spr
runtime       : 48:00:00
system        : stampede
num_nodes     : 2
exclude_time  : 10
save_interval : 10
base_dir      : /scratch/09909/smata/wrf_les_sweep/runs/10MW/rate/OOS_SV
wrf_path      : /work2/09909/smata/stampede3/WRF_LES
template_path : /scratch/09909/smata/wrf_les_sweep/templates
turb_model    : iea10MW
rotor_model   : GAD
slice_loc     : 1
print_table   : True
plot_outer    : True
save_outer    : False
outer_align   : False
plot_inner    : True
save_inner    : False
save_both     : False
outer_pad     : 75
plot_profiles : True
save_profiles : True
batch_submit  : True
prof_type     : Idealized
shear_type    : Rate
shear         : [-0.035, -0.025, -0.015, -0.005, 0, 0.005, 0.015, 0.025, 0.035]
veer          : [-0.25, -0.15, -0.075, -0.035, -0.01, 0.01, 0.035, 0.075, 0.15, 0.25]
Ufst          : 7


In [17]:
cases = [pair for pair in product(params['shear'], params['veer'])]
shears = [pair[0] for pair in cases]
veers = [pair[1] for pair in cases]

shears = np.array(shears)  
veers  = np.array(veers)

In [18]:
Nelm = wrfles[0]['Nelm']
Nsct = wrfles[0]['Nsct']

t = np.linspace(0,2*np.pi,Nsct)
r = np.linspace(0,1,Nelm)

rho = 1.225

z_hh = wrfles[0]['hub_height']

R, T = np.meshgrid(r, t)

X = R * np.sin(T)
Y = (R * np.cos(T)) * wrfles[0]['radius'] + z_hh

wrf_vax      = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_vtn      = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_vax_real = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_vtn_real = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_vtn_NR   = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_vtn_NR_real= np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_W        = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_W_real   = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_CL       = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_CD       = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_CL_real  = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_CD_real  = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_L        = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_D        = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_L_real   = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_D_real   = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_FN       = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_FT       = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_FN_real  = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_FT_real  = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_phi      = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_phi_real = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_aoa      = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_aoa_real = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_cax      = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

wrf_pow      = np.zeros(len(casenames),dtype='longdouble')
wrf_thr      = np.zeros(len(casenames),dtype='longdouble')

wrf_pow_real = np.zeros(len(casenames),dtype='longdouble')
wrf_thr_real = np.zeros(len(casenames),dtype='longdouble')

shapiro      = np.zeros(len(casenames), dtype=float)

Uhub         = np.zeros(len(casenames), dtype=float)

wrf_tsr      = np.zeros(len(casenames), dtype=float)
wrf_omg      = np.zeros(len(casenames), dtype=float)

wrf_cot_rot  = np.zeros(len(casenames), dtype=float)
wrf_ind_rot  = np.zeros(len(casenames), dtype=float)

r_ann        = np.zeros((Nelm, len(casenames)),dtype='longdouble')
shears_ann   = np.zeros((Nelm, len(casenames)),dtype='longdouble')
veers_ann    = np.zeros((Nelm, len(casenames)),dtype='longdouble')

wrf_cot_ann  = np.zeros((Nelm, len(casenames)),dtype='longdouble')
wrf_ind_ann  = np.zeros((Nelm, len(casenames)),dtype='longdouble')

wrf_cot_loc  = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')
wrf_ind_loc  = np.zeros((Nelm, Nsct, len(casenames)),dtype='longdouble')

rotor = IEA10MW()

for count,case in enumerate(casenames):

    # Rotor radius
    R = wrfles[count]['radius']

    # Extract background flow
    u_func = interp1d(data[case][:,0], data[case][:,1], kind='linear')
    v_func = interp1d(data[case][:,0], data[case][:,2], kind='linear')

    # u and v velocity functions
    u_inf = u_func(Y)
    v_inf = v_func(Y)

    # Extract u and v velocity components at the rotor disk
    # u_rotor = np.mean(wrfles[count]['shapiroM'][0] * wrfles[count]['u'],axis=0)
    u_rotor = np.mean(wrfles[count]['u'],axis=0)
    v_rotor = np.mean(wrfles[count]['v'],axis=0)
    w_rotor = np.mean(wrfles[count]['w'],axis=0)

    # Magnitude of wind speed at rotor disk
    U = np.sqrt(u_rotor**2 + v_rotor**2)

    # Local wind misalignment angle at the rotor disk
    wdir = np.atan2(v_rotor,u_rotor)

    # Nondimensional radial positions
    r = wrfles[count]['rOverR']
    r_mat =  (np.ones_like(wdir.T) * r).T

    # Azimuthal coordinates
    theta = np.linspace(0, 2*np.pi, wrfles[count]['Nsct'])

    # Freestream wind speed at hub height
    U_hub = np.sqrt(u_func(z_hh)**2 + v_func(z_hh)**2)

    # Radial chord lengths
    chord = rotor.chord_func(r)
    chord =  (np.ones_like(wdir.T) * chord).T

    # Radial twist
    twist = rotor.twist_func(r)
    
    # Local solidity
    sigma_r = 3 * chord / (2 * np.pi * r_mat * R)

    # Local solidity
    sigma = 3/Nsct

    # Tip speed ratio
    omega = np.mean(wrfles[count]['omega'], axis=0) * 2 * np.pi / 60
    tsr = np.mean(omega*R/(U_hub), axis=0) 

    trbYaw = np.mean(wrfles[count]['yaw'], axis=0) 

    Vax, Vtn_NR, _ = postproc.rotGlobalToLocal(Nelm,Nsct,u_rotor,v_rotor,np.zeros_like(u_rotor))

    Vtn = omega * r_mat * R - Vtn_NR

    # Relative velocity
    W = np.sqrt(Vax**2 + Vtn**2)

    # Inflow angle
    phi = np.atan2(Vax,Vtn)

    aoa  = phi - twist[:, np.newaxis]
    Cl, Cd = rotor.clcd(r_mat, aoa)

    # Axial coefficient
    Cax = Cl * np.cos(phi) + Cd * np.sin(phi)

    # Local CT
    ct = sigma_r * (W/U_hub)**2 * Cax

    L = 1/2 * rho * chord * (Cl * W**2)
    D = 1/2 * rho * chord * (Cd * W**2)

    FN = L * np.cos(phi) + D * np.sin(phi)
    FT = L * np.sin(phi) - D * np.cos(phi)

    dr = (R - rotor.hub_radius)/Nelm

    T = np.sum(FN * dr * sigma)
    P = np.sum(FT * r_mat * R * dr * sigma * omega)

    wrf_vax[:,:,count]      = Vax
    wrf_vtn[:,:,count]      = Vtn

    wrf_vax_real[:,:,count] = np.mean(wrfles[count]['v1'], axis=0)
    wrf_vtn_real[:,:,count] = np.mean(wrfles[count]['v_tan'], axis=0)
  
    wrf_vtn_NR[:,:,count]   = Vtn_NR
    wrf_vtn_NR_real[:,:,count] = np.mean(wrfles[count]['v_tan_no_rot'], axis=0)

    wrf_phi[:,:,count]      = phi
    wrf_phi_real[:,:,count] = np.deg2rad(np.mean(wrfles[count]['phi'], axis=0))

    wrf_aoa[:,:,count]      = aoa
    wrf_aoa_real[:,:,count] = np.deg2rad(np.mean(wrfles[count]['aoa'], axis=0))

    wrf_W[:,:,count]        = W
    wrf_W_real[:,:,count]   = np.mean(wrfles[count]['vrel'], axis=0)

    wrf_cax[:,:,count]      = Cax

    wrf_cot_rot[count]      = postproc.rotor_average(ct,r,theta)
    wrf_ind_rot[count]      = postproc.rotor_average(1 - u_rotor/ U_hub,r,theta)

    wrf_cot_ann[:,count]    = postproc.annulus_average(ct,theta)
    wrf_ind_ann[:,count]    = postproc.annulus_average(1 - u_rotor/ U_hub,theta)

    r_ann[:,count]          = r
    shears_ann[:,count]     = shears[count] * np.ones_like(r)
    veers_ann[:,count]      = veers[count] * np.ones_like(r)

    wrf_cot_loc[:,:,count]  = ct
    wrf_ind_loc[:,:,count]  = 1 - u_rotor/ U_hub

    wrf_CL[:,:,count]       = Cl
    wrf_CD[:,:,count]       = Cd

    wrf_CL_real[:,:,count]  = np.mean(wrfles[count]['cl'], axis=0)
    wrf_CD_real[:,:,count]  = np.mean(wrfles[count]['cd'], axis=0)

    wrf_L[:,:,count]        = L
    wrf_D[:,:,count]        = D

    wrf_L_real[:,:,count]   = np.mean(wrfles[count]['l'], axis=0)
    wrf_D_real[:,:,count]   = np.mean(wrfles[count]['d'], axis=0)

    wrf_FN[:,:,count]       = FN
    wrf_FT[:,:,count]       = FT

    wrf_FN_real[:,:,count]  = np.mean(wrfles[count]['fn'], axis=0)
    wrf_FT_real[:,:,count]  = np.mean(wrfles[count]['ft'], axis=0)

    wrf_thr[count]          = T
    wrf_pow[count]          = P

    wrf_thr_real[count]     = np.mean(wrfles[count]['thrust'], axis=0)[0]
    wrf_pow_real[count]     = np.mean(wrfles[count]['power_aero'], axis=0)[0]

    Uhub[count]             = U_hub
    wrf_tsr[count]          = tsr
    wrf_omg[count]          = omega[0]

# END LOOP

np.save(save_path + 'wrf_cot_loc.npy', wrf_cot_loc)
np.save(save_path + 'wrf_ind_loc.npy', wrf_ind_loc)

np.save(save_path + 'wrf_cot_ann.npy', wrf_cot_ann)
np.save(save_path + 'wrf_ind_ann.npy', wrf_ind_ann)

np.save(save_path + 'wrf_cot_rot.npy', wrf_cot_rot)
np.save(save_path + 'wrf_ind_rot.npy', wrf_ind_rot)
np.savez(save_path + "rotor_dims_10MW.npz", Nsct=Nsct, Nelm=Nelm, rOverR=r, R = R)

In [19]:
# Create a list of (name, array) pairs
variables = [
    ('r_ann'      , r_ann),
    ('wrf_cot_loc', wrf_cot_loc),
    ('wrf_ind_loc', wrf_ind_loc),
    ('wrf_cot_ann', wrf_cot_ann),
    ('wrf_ind_ann', wrf_ind_ann),
    ('wrf_cot_rot', wrf_cot_rot),
    ('wrf_ind_rot', wrf_ind_rot),
    ('shears'     , shears),
    ('veers'      , veers),
    ('shears_ann' , shears_ann),
    ('veers_ann'  , veers_ann),
]

# Print descriptive statistics
print("Variable Statistics:")
print("-" * 75)
print(f"{'Name':<15} | {'Mean':>8} | {'Std Dev':>8} | {'Min':>8} | {'Max':>8}")
print("-" * 75)

for name, arr in variables:
    arr_flat = arr.flatten()
    mean = np.mean(arr_flat)
    std = np.std(arr_flat)
    min_val = np.min(arr_flat)
    max_val = np.max(arr_flat)
    print(f"{name:<15} | {mean:>8.4f} | {std:>8.4f} | {min_val:>8.4f} | {max_val:>8.4f}")

Variable Statistics:
---------------------------------------------------------------------------
Name            |     Mean |  Std Dev |      Min |      Max
---------------------------------------------------------------------------
r_ann           |   0.5121 |   0.2815 |   0.0429 |   0.9812
wrf_cot_loc     |   0.7181 |   0.3140 |   0.0679 |   1.2575
wrf_ind_loc     |   0.2306 |   0.1625 |  -0.4475 |   0.7032
wrf_cot_ann     |   0.7181 |   0.3033 |   0.1032 |   1.0552
wrf_ind_ann     |   0.2306 |   0.1102 |  -0.0171 |   0.3567
wrf_cot_rot     |   0.6930 |   0.0086 |   0.6726 |   0.7116
wrf_ind_rot     |   0.2354 |   0.0126 |   0.2111 |   0.2647
shears          |  -0.0000 |   0.0216 |  -0.0350 |   0.0350
veers           |   0.0000 |   0.1356 |  -0.2500 |   0.2500
shears_ann      |   0.0000 |   0.0216 |  -0.0350 |   0.0350
veers_ann       |   0.0000 |   0.1356 |  -0.2500 |   0.2500


In [21]:
# Generate MATLAB tables of standardized inputs

X_rot = np.column_stack([wrf_cot_rot, shears, veers])
X_ann = np.column_stack([r_ann.flatten(), wrf_cot_ann.flatten(), shears_ann.flatten(), veers_ann.flatten()])

y_rot = wrf_ind_rot
y_ann = wrf_ind_ann.flatten()

savemat(save_path + 'wrf_10MW_ann_RAW.mat', {'X': X_ann, 'y': y_ann.reshape(-1, 1)})
savemat(save_path + 'wrf_10MW_rot_RAW.mat', {'X': X_rot, 'y': y_rot.reshape(-1, 1)})

In [23]:
# ── standardize every array WITH TRAINING SCALARS, overwriting in place ──────────────────────────────
# 1-D (_rot) -------------------------------------------------------------------
# scaler_wrf_cot_rot = StandardScaler()
wrf_cot_rot = scaler_wrf_cot_rot.transform(wrf_cot_rot[:, None]).ravel()

# scaler_wrf_ind_rot = StandardScaler()
wrf_ind_rot = scaler_wrf_ind_rot.transform(wrf_ind_rot[:, None]).ravel()

# scaler_shears = StandardScaler()
shears = scaler_shears.transform(shears[:, None]).ravel()

# scaler_veers = StandardScaler()
veers = scaler_veers.transform(veers[:, None]).ravel()

# 2-D (_ann) -------------------------------------------------------------------
# scaler_wrf_cot_ann = StandardScaler()
wrf_cot_ann = scaler_wrf_cot_ann.transform(wrf_cot_ann.flatten()[:, None]) \
                                .reshape(wrf_cot_ann.shape)

# scaler_wrf_ind_ann = StandardScaler()
wrf_ind_ann = scaler_wrf_ind_ann.transform(wrf_ind_ann.flatten()[:, None]) \
                                .reshape(wrf_ind_ann.shape)

# scaler_shears_ann = StandardScaler()
shears_ann = scaler_shears_ann.transform(shears_ann.flatten()[:, None]) \
                              .reshape(shears_ann.shape)

# scaler_veers_ann = StandardScaler()
veers_ann = scaler_veers_ann.transform(veers_ann.flatten()[:, None]) \
                            .reshape(veers_ann.shape)

In [24]:
# Create a list of (name, array) pairs
variables = [
    ('r_ann'      , r_ann),
    ('wrf_cot_loc', wrf_cot_loc),
    ('wrf_ind_loc', wrf_ind_loc),
    ('wrf_cot_ann', wrf_cot_ann),
    ('wrf_ind_ann', wrf_ind_ann),
    ('wrf_cot_rot', wrf_cot_rot),
    ('wrf_ind_rot', wrf_ind_rot),
    ('shears'     , shears),
    ('veers'      , veers),
    ('shears_ann' , shears_ann),
    ('veers_ann'  , veers_ann),
]

# Print descriptive statistics
print("Variable Statistics:")
print("-" * 75)
print(f"{'Name':<15} | {'Mean':>8} | {'Std Dev':>8} | {'Min':>8} | {'Max':>8}")
print("-" * 75)

for name, arr in variables:
    arr_flat = arr.flatten()
    mean = np.mean(arr_flat)
    std = np.std(arr_flat)
    min_val = np.min(arr_flat)
    max_val = np.max(arr_flat)
    print(f"{name:<15} | {mean:>8.4f} | {std:>8.4f} | {min_val:>8.4f} | {max_val:>8.4f}")

Variable Statistics:
---------------------------------------------------------------------------
Name            |     Mean |  Std Dev |      Min |      Max
---------------------------------------------------------------------------
r_ann           |   0.5121 |   0.2815 |   0.0429 |   0.9812
wrf_cot_loc     |   0.7181 |   0.3140 |   0.0679 |   1.2575
wrf_ind_loc     |   0.2306 |   0.1625 |  -0.4475 |   0.7032
wrf_cot_ann     |   0.0209 |   1.0096 |  -2.0260 |   1.1430
wrf_ind_ann     |  -0.0774 |   1.0121 |  -2.3532 |   1.0812
wrf_cot_rot     |   0.8366 |   1.2488 |  -2.1443 |   3.5422
wrf_ind_rot     |  -0.8444 |   1.2539 |  -3.2666 |   2.0761
shears          |   0.0000 |   1.0801 |  -1.7500 |   1.7500
veers           |   0.0000 |   1.2481 |  -2.3009 |   2.3009
shears_ann      |   0.0000 |   1.0801 |  -1.7500 |   1.7500
veers_ann       |   0.0000 |   1.2481 |  -2.3009 |   2.3009


In [ ]:
# # Save scalers

# with open(os.path.join(save_path, 'scaler_wrf_cot_ann.pkl'), 'wb') as f:
#     pickle.dump(scaler_wrf_cot_ann, f)

# with open(os.path.join(save_path, 'scaler_wrf_ind_ann.pkl'), 'wb') as f:
#     pickle.dump(scaler_wrf_ind_ann, f)

# with open(os.path.join(save_path, 'scaler_wrf_cot_rot.pkl'), 'wb') as f:
#     pickle.dump(scaler_wrf_cot_rot, f)

# with open(os.path.join(save_path, 'scaler_wrf_ind_rot.pkl'), 'wb') as f:
#     pickle.dump(scaler_wrf_ind_rot, f)

# with open(os.path.join(save_path, 'scaler_shears_rot.pkl'), 'wb') as f:
#     pickle.dump(scaler_shears, f)

# with open(os.path.join(save_path, 'scaler_veers_rot.pkl'), 'wb') as f:
#     pickle.dump(scaler_veers, f)

# with open(os.path.join(save_path, 'scaler_shears_ann.pkl'), 'wb') as f:
#     pickle.dump(scaler_shears_ann, f)

# with open(os.path.join(save_path, 'scaler_veers_ann.pkl'), 'wb') as f:
#     pickle.dump(scaler_veers_ann, f)

In [25]:
# Generate MATLAB tables of standardized inputs

X_rot = np.column_stack([wrf_cot_rot, shears, veers])
X_ann = np.column_stack([r_ann.flatten(), wrf_cot_ann.flatten(), shears_ann.flatten(), veers_ann.flatten()])

y_rot = wrf_ind_rot
y_ann = wrf_ind_ann.flatten()

savemat(save_path + 'wrf_10MW_ann.mat', {'X': X_ann, 'y': y_ann.reshape(-1, 1)})
savemat(save_path + 'wrf_10MW_rot.mat', {'X': X_rot, 'y': y_rot.reshape(-1, 1)})